# Code 14.3.2: Likelihood displacement when responses share a direction

The chosen and rejected responses have nearly aligned score vectors, and a third response points somewhere else. The DPO margin rises while both labeled responses lose probability.


In [ ]:
import torch
import torch.nn.functional as F

v_w = torch.tensor([1.00, 0.10])
v_l = torch.tensor([0.95, -0.10])
v_o = torch.tensor([-0.20, 1.00])
theta = torch.tensor([0.30, 0.00], requires_grad=True)
opt = torch.optim.SGD([theta], lr=0.4)
beta = 0.5

scores = torch.stack([v_w @ theta, v_l @ theta, v_o @ theta])
ref = F.log_softmax(scores.detach(), dim=0)

for step in range(8):
    scores = torch.stack([v_w @ theta, v_l @ theta, v_o @ theta])
    logp = F.log_softmax(scores, dim=0)
    h = (logp[0] - ref[0]) - (logp[1] - ref[1])
    loss = -F.logsigmoid(beta * h)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step in (0, 7):
        print(f"step {step}: loss {loss.item():.4f}  "
              f"logp {[round(v, 3) for v in logp.tolist()]}  h {h.item():.3f}")


**Expected output**

Output:

```text
step 0: loss 0.6931  logp [-0.987, -1.002, -1.347]  h 0.000
step 7: loss 0.6858  logp [-0.999, -1.043, -1.275]  h 0.030
```

At step 7 the numbers are the state *before* the eighth update, which is the state after seven updates. The margin $h$ grew. The log-probabilities of both the chosen response (first entry) and the rejected response (second) fell, and the unlabeled response rose.
